# Byte Pair Encoding (BPE) Tokenizer

Tokenization is the first step in every modern LLM pipeline: raw text is converted into a sequence of integer *token IDs* that the model processes. This notebook builds a **BPE tokenizer from scratch** in pure Python and analyzes it on the TinyStories corpus.

**Why this matters:**
- The previous course (DataSciPy) used *character-level* tokenization — one id per character.
- Real LLMs use *subword* tokenization. GPT-2/GPT-4 use BPE; LLaMA uses SentencePiece (also BPE-based).
- BPE lets the model see longer contexts with the same sequence length because common substrings are compressed into single tokens.

**What we build:**
1. BPE training algorithm (merge-based vocabulary construction)
2. Encode and decode functions
3. Vocabulary analysis — token frequency, length distribution
4. Compression ratio: how much shorter are token sequences vs character sequences?
5. Save the trained tokenizer to disk for use in `nanochat.ipynb`

In [ ]:
import re
import json
import pickle
import urllib.request
import collections
import matplotlib.pyplot as plt
import numpy as np

## 1. The BPE algorithm

**Byte Pair Encoding** (Sennrich et al., 2016) was originally a text *compression* algorithm. Applied to tokenization:

1. Start with a vocabulary of all **individual characters** in the training corpus.
2. Count every adjacent pair of symbols.
3. Merge the most frequent pair into a new token, replacing all occurrences in the corpus.
4. Repeat until the vocabulary reaches the target size.

### Toy example

Corpus: `"low low low lower lowest"`

| Step | Merge | Vocab | Tokens |
|------|-------|-------|--------|
| 0 | — | `{l,o,w,e,r,s,t,' '}` | `l o w   l o w   …` |
| 1 | `l+o→lo` | adds `lo` | `lo w   lo w   …` |
| 2 | `lo+w→low` | adds `low` | `low   low   …` |
| 3 | `low+e→lowe` | adds `lowe` | `low   low   lowe r   lowe st` |

Each merge step reduces total token count while adding one vocabulary entry. The algorithm stops when vocabulary size reaches the target (e.g., 512 or 50,000).

### Character-level vs BPE

| Property | Char-level | BPE |
|----------|-----------|-----|
| Vocab size | ~70 | 512 – 100k |
| Sequence length | long | shorter |
| Out-of-vocab | impossible | impossible (falls back to chars) |
| Common subwords | split across tokens | one token |
| Morphology | lost | partially preserved |

## 2. Implementation

In [ ]:
class BPETokenizer:
    """
    Character-level BPE tokenizer (not byte-level).
    Trains on raw text; encodes/decodes using the learned merge rules.
    """

    def __init__(self):
        self.vocab = []      # list of token strings, index = token id
        self.encoder = {}    # token string → token id
        self.merges = []     # list of (id_a, id_b) merge rules in order
        self.vocab_size = 0

    # ── Training ──────────────────────────────────────────────────────────────

    def train(self, text, vocab_size=512, verbose=True):
        """Build vocabulary from text using BPE merge rules."""
        # Initialise with all unique characters
        chars = sorted(set(text))
        self.vocab = chars[:]
        self.encoder = {c: i for i, c in enumerate(chars)}
        ids = [self.encoder[c] for c in text]
        self.merges = []

        while len(self.vocab) < vocab_size:
            # Count adjacent pairs
            counts = {}
            for a, b in zip(ids, ids[1:]):
                counts[(a, b)] = counts.get((a, b), 0) + 1
            if not counts:
                break

            # Merge the most frequent pair
            best = max(counts, key=counts.get)
            new_id = len(self.vocab)
            new_tok = self.vocab[best[0]] + self.vocab[best[1]]
            self.vocab.append(new_tok)
            self.encoder[new_tok] = new_id
            self.merges.append(best)

            # Apply merge to the token stream
            merged, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and (ids[i], ids[i + 1]) == best:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(ids[i])
                    i += 1
            ids = merged

            if verbose and len(self.vocab) % 64 == 0:
                print(f"  vocab={len(self.vocab):4d}  tokens={len(ids):,}")

        self.vocab_size = len(self.vocab)
        if verbose:
            print(f"Training complete: vocab_size={self.vocab_size}, "
                  f"tokens={len(ids):,} (compression ratio "
                  f"{len(text)/len(ids):.2f}×)")

    # ── Encoding ──────────────────────────────────────────────────────────────

    def encode(self, text):
        """Convert text to list of token ids."""
        ids = [self.encoder.get(c, 0) for c in text]
        for a, b in self.merges:
            new_id = self.encoder[self.vocab[a] + self.vocab[b]]
            merged, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(ids[i])
                    i += 1
            ids = merged
        return ids

    # ── Decoding ──────────────────────────────────────────────────────────────

    def decode(self, ids):
        """Convert list of token ids back to text."""
        return ''.join(self.vocab[i] for i in ids if 0 <= i < len(self.vocab))

    # ── Serialisation ─────────────────────────────────────────────────────────

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({'vocab': self.vocab, 'merges': self.merges}, f)
        print(f"Saved tokenizer to {path}")

    @classmethod
    def load(cls, path):
        with open(path, 'rb') as f:
            state = pickle.load(f)
        tok = cls()
        tok.vocab = state['vocab']
        tok.merges = [tuple(m) for m in state['merges']]
        tok.encoder = {t: i for i, t in enumerate(tok.vocab)}
        tok.vocab_size = len(tok.vocab)
        return tok

## 3. Toy example: trace the merges

Verify on a small string so we can inspect every step.

In [ ]:
toy = "low low low lower lowest"
tok_toy = BPETokenizer()
tok_toy.train(toy, vocab_size=15, verbose=False)

print("Merge rules learned:")
for i, (a, b) in enumerate(tok_toy.merges):
    print(f"  {i+1:2d}: '{tok_toy.vocab[a]}' + '{tok_toy.vocab[b]}' → '{tok_toy.vocab[a]+tok_toy.vocab[b]}'")

print(f"\nFull vocabulary ({tok_toy.vocab_size} tokens):")
print(tok_toy.vocab)

print("\nEncoding 'lowest':")
ids = tok_toy.encode('lowest')
tokens = [tok_toy.vocab[i] for i in ids]
print(f"  ids:    {ids}")
print(f"  tokens: {tokens}")
print(f"  decoded: {tok_toy.decode(ids)!r}")

## 4. Train on TinyStories

Download the validation split of TinyStories (~10 MB) and train a 512-token vocabulary on it.

In [ ]:
URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt"
CACHE = "TinyStoriesV2-GPT4-valid.txt"

import os, urllib.request
if not os.path.exists(CACHE):
    print(f"Downloading TinyStories validation split …")
    urllib.request.urlretrieve(URL, CACHE)
    print("Done.")

with open(CACHE, encoding='utf-8') as f:
    raw = f.read()

# Use the first MAX_STORIES stories (separated by <|endoftext|>)
MAX_STORIES = 5000
stories = [s.strip() for s in raw.split('<|endoftext|>') if s.strip()][:MAX_STORIES]
text = '\n'.join(stories)
print(f"Stories: {len(stories):,}  |  Characters: {len(text):,}")

In [ ]:
VOCAB_SIZE = 512
tokenizer = BPETokenizer()
tokenizer.train(text, vocab_size=VOCAB_SIZE)

## 5. Vocabulary analysis

Inspect what the tokenizer learned: token length distribution, the longest tokens, and which single characters exist in the vocabulary.

In [ ]:
token_lengths = [len(t) for t in tokenizer.vocab]

fig, axes = plt.subplots(1, 2, figsize=(11, 3))

axes[0].bar(range(1, max(token_lengths)+1),
            [token_lengths.count(l) for l in range(1, max(token_lengths)+1)],
            color='steelblue', edgecolor='white')
axes[0].set_xlabel('Token length (characters)')
axes[0].set_ylabel('Count')
axes[0].set_title('Vocabulary: token length distribution')

# Show the 20 longest tokens
longest = sorted(tokenizer.vocab, key=len, reverse=True)[:20]
axes[1].barh(range(len(longest)), [len(t) for t in longest], color='salmon')
axes[1].set_yticks(range(len(longest)))
axes[1].set_yticklabels([repr(t) for t in longest], fontsize=8)
axes[1].set_xlabel('Token length')
axes[1].set_title('20 longest tokens')

plt.tight_layout()
plt.show()

print(f"\nCharacter tokens (len=1): {sum(1 for t in tokenizer.vocab if len(t)==1)}")
print(f"Multi-char tokens:         {sum(1 for t in tokenizer.vocab if len(t)>1)}")
print(f"Longest token: {repr(max(tokenizer.vocab, key=len))}")

## 6. Compression ratio

How many fewer tokens does BPE produce compared to a character-level representation?

$$\text{compression ratio} = \frac{\text{characters}}{\text{BPE tokens}}$$

A ratio of 2 means each BPE token represents 2 characters on average — the model sees 2× more context within the same sequence length.

In [ ]:
# Measure compression on a held-out sample
sample_stories = stories[:200]
sample_text = '\n'.join(sample_stories)

char_tokens = len(sample_text)
bpe_tokens = len(tokenizer.encode(sample_text))

ratio = char_tokens / bpe_tokens
print(f"Characters:         {char_tokens:,}")
print(f"BPE tokens:         {bpe_tokens:,}")
print(f"Compression ratio:  {ratio:.2f}×")
print(f"Avg token length:   {ratio:.2f} chars/token")

# Plot compression vs vocab size for different checkpoints
# (rerun training at multiple vocab sizes to measure)
vocab_sizes = [64, 128, 256, 512]
ratios = []
for vs in vocab_sizes:
    t = BPETokenizer()
    t.train(text, vocab_size=vs, verbose=False)
    enc = t.encode(sample_text)
    ratios.append(len(sample_text) / len(enc))
    print(f"  vocab={vs:4d}  ratio={ratios[-1]:.2f}×")

plt.figure(figsize=(6, 3))
plt.plot(vocab_sizes, ratios, marker='o', color='steelblue')
plt.xlabel('Vocabulary size')
plt.ylabel('Compression ratio')
plt.title('BPE compression ratio vs vocabulary size')
plt.tight_layout()
plt.show()

## 7. Tokenize some examples

See how the tokenizer segments different kinds of text.

In [ ]:
examples = [
    "Once upon a time there was a little cat.",
    "The princess lived in a tall castle.",
    "abcdefghijklmnopqrstuvwxyz",   # unknown sequences
]

for ex in examples:
    ids = tokenizer.encode(ex)
    toks = [tokenizer.vocab[i] for i in ids]
    print(f"Input:  {ex!r}")
    print(f"Tokens: {toks}")
    print(f"IDs:    {ids}")
    print(f"Ratio:  {len(ex)/len(ids):.2f}×")
    print()

## 8. Save the tokenizer

Save the trained tokenizer to `bpe_tokenizer.pkl`. This file is loaded by `nanochat.ipynb` so we don't need to retrain every time.

In [ ]:
tokenizer.save('bpe_tokenizer.pkl')

# Verify round-trip
tok2 = BPETokenizer.load('bpe_tokenizer.pkl')
test = "Once upon a time there was a fox."
assert tok2.decode(tok2.encode(test)) == test
print(f"Round-trip OK: {test!r}")

## 9. Exercises

1. **Vocabulary size ablation.** Train tokenizers with vocab sizes 64, 128, 256, 512, 1024. Plot compression ratio and the 10 longest tokens for each. Where does increasing vocab size stop helping?

2. **Byte-level BPE.** Modify `train()` to operate on UTF-8 *bytes* rather than characters (encode each character as `c.encode('utf-8')` first). How does this change the vocabulary for text with accented characters?

3. **Merge order.** Print the first 20 and last 20 merge rules. What patterns do you see? Why are spaces merged early?

4. **Out-of-distribution text.** Encode a sentence in another language (e.g., Spanish or French). Count how many tokens fall back to character-level. How does the compression ratio compare?

5. **Tokenization ambiguity.** Is BPE encoding deterministic? Can the same string be tokenized in multiple ways? Why or why not?

6. **Token frequency.** Encode the full corpus and count token frequencies. Plot a Zipf plot (rank vs frequency on log-log axes). Does token frequency follow a power law?

## References

- Sennrich, Haddow & Birch (2016). [Neural Machine Translation of Rare Words with Subword Units](https://arxiv.org/abs/1508.07909). *ACL 2016. Introduces BPE for NLP.*
- Radford et al. (2019). [Language Models are Unsupervised Multitask Learners](https://openai.com/research/language-unsupervised). *GPT-2: uses byte-level BPE with 50,257 vocab.*
- Karpathy (2023). [Let's build the GPT tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE). *Video walkthrough of BPE from scratch.*
- Karpathy (2025). [nanochat](https://github.com/karpathy/nanochat). *Uses a Rust BPE tokenizer with 64k vocab on FineWeb-EDU.*